In [5]:
!pip install -U transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


## Local Inference on GPU
Model page: https://huggingface.co/google/flan-t5-xl

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/flan-t5-xl)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-xl")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-xl")

config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [1]:
import sys
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re
# import bitsandbytes # Removed as per user request
# import accelerate # Removed as per user request

##### You may comment this section to see verbose -- but you must un-comment this before final submission. ######
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
#################################################################################################################

"""
* * * Changes allowed from here  * * *
"""

def llm_function(model,tokenizer,questions):
    '''
    The steps are given for your reference:

    1. Generate answer for the first question.
    2. Generate answer for the second question use the answer for first question as context.
    3. Generate a deterministic output either 'YES' or 'NO' for the third question using the context from second question.
    5. Clean output and return.
    6. Output is case-sensative: YES or NO
    Note: The model (Flan-T5-XL) and tokenizer is already initialized. Do not modify that section.
    '''
    question_a, question_b, question_c = questions
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model.to(device) # Move model to GPU if available - no longer needed with device_map

    # Define a default max_new_tokens for general answers to suppress warnings
    # Reduced max_new_tokens to further optimize memory for first two questions
    default_max_new_tokens = 30

    # 1. Generate answer for the first question.
    # Re-added .to(device) to explicitly move input to GPU for compatibility with device_map="auto"
    input_a = tokenizer(question_a, return_tensors='pt').input_ids.to(device)
    output_a = model.generate(input_a, max_new_tokens=default_max_new_tokens)
    answer_a = tokenizer.decode(output_a[0], skip_special_tokens=True)

    # 2. Generate answer for the second question use the answer for first question as context.
    prompt_b = f"{question_b} Context: {answer_a}"
    # Re-added .to(device) to explicitly move input to GPU for compatibility with device_map="auto"
    input_b = tokenizer(prompt_b, return_tensors='pt').input_ids.to(device)
    output_b = model.generate(input_b, max_new_tokens=default_max_new_tokens)
    answer_b = tokenizer.decode(output_b[0], skip_special_tokens=True)

    # 3. Generate a deterministic output either 'YES' or 'NO' for the third question using the context from second question.
    prompt_c = f"{question_c} Context: {answer_b} Answer only with YES or NO."
    # Re-added .to(device) to explicitly move input to GPU for compatibility with device_map="auto"
    input_c = tokenizer(prompt_c, return_tensors='pt').input_ids.to(device)
    # Using max_new_tokens to limit output length, and num_beams for potentially better quality
    output_c = model.generate(input_c, max_new_tokens=5, num_beams=5, early_stopping=True)
    answer_c = tokenizer.decode(output_c[0], skip_special_tokens=True)

    # 4. Clean output and return.
    # 5. Output is case-sensative: YES or NO
    final_output = answer_c.strip().upper()

    # Ensure the output is strictly 'YES' or 'NO' based on the problem statement.
    # The prompt "Answer only with YES or NO." should guide the LLM.
    # If the LLM still produces variations, we enforce the "YES" or "NO" rule.
    if "YES" in final_output:
        final_output = "YES"
    elif "NO" in final_output:
        final_output = "NO"
    else:
        # Default to 'NO' if the model's output doesn't clearly contain 'YES' or 'NO',
        # to ensure the function always returns one of the required values.
        final_output = "NO"

    return final_output

"""
ALERT: * * * No changes are allowed below this comment  * * *
"""

if __name__ == '__main__':

    # The following lines are commented out because sys.argv expects command-line arguments,
    # which are not typically provided when running directly in a Colab cell.
    # IndexError: list index out of range occurs because sys.argv does not contain enough elements.


    # question_a = "Who is Rabindranath Tagore?"
    # question_b = "Where was he born?"
    # question_c = "Is it in America?"

    question_a = sys.argv[1].strip()
    question_b = sys.argv[2].strip()
    question_c = sys.argv[3].strip()

    questions = [question_a, question_b, question_c]
    ##################### Loading Model and Tokenizer ########################
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl")
    # Loading model in half-precision (FP16) to reduce memory usage without 8-bit quantization
    model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl", torch_dtype=torch.float16, device_map="auto")
    ##########################################################################

    """  Call to function that will perform the computation. """
    torch.manual_seed(42)
    out = llm_function(model,tokenizer,questions)
    print(out.strip())


NO
